In [1]:
import json
import time

import joblib
import numpy as np
import pandas as pd
import torch

from benchmark.config import DATA_PROCESSED, DATASETS, MODELS_DIR, RESULTS_BENCHMARK
from benchmark.metrics import evaluate_regression
from benchmark.models.xrfm_model import make_processor
from benchmark.models.xrfm_model import fit_best_xrfm, fit_xrfm_cross_validation

data_name = "superconductor"
cfg = DATASETS[data_name]
processed = DATA_PROCESSED / data_name
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
train_df = pd.read_csv(processed / "train.csv")
train_df.head()

,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,...,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
0,2,61.785750,56.797395,60.826648,56.061368,0.677665,0.644466,21.688500,17.577195,10.844250,...,4.730000,4.472136,4.707651,0.686962,0.537250,1,2.570000,0.500000,0.443959,6.60
1,3,47.310532,51.865118,43.807482,49.246493,1.032302,0.828781,34.628195,30.002880,16.267662,...,2.412000,2.519842,2.306971,1.039721,1.013695,2,0.788000,0.942809,0.808861,6.70
2,4,76.517718,56.808817,59.310096,35.773432,1.197273,0.981880,122.906070,34.833160,44.289459,...,2.264286,2.213364,2.226222,1.368922,1.048834,1,1.100000,0.433013,0.440952,34.50
3,4,137.184413,105.454381,129.327237,95.279217,1.333014,0.937968,122.454000,50.444024,43.422607,...,4.470588,4.426728,4.400411,1.368922,0.826315,2,2.800000,0.866025,0.848365,4.11
4,5,98.562254,57.854867,71.960198,35.849123,1.364108,0.958288,192.981000,38.095568,67.769561,...,2.296067,2.605171,2.250329,1.531778,1.042065,3,1.151276,1.166190,0.500706,22.70


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17010 entries, 0 to 17009
Data columns (total 82 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   number_of_elements               17010 non-null  int64  
 1   mean_atomic_mass                 17010 non-null  float64
 2   wtd_mean_atomic_mass             17010 non-null  float64
 3   gmean_atomic_mass                17010 non-null  float64
 4   wtd_gmean_atomic_mass            17010 non-null  float64
 5   entropy_atomic_mass              17010 non-null  float64
 6   wtd_entropy_atomic_mass          17010 non-null  float64
 7   range_atomic_mass                17010 non-null  float64
 8   wtd_range_atomic_mass            17010 non-null  float64
 9   std_atomic_mass                  17010 non-null  float64
 10  wtd_std_atomic_mass              17010 non-null  float64
 11  mean_fie                         17010 non-null  float64
 12  wtd_mean_fie      

In [4]:
test_df = pd.read_csv(processed / "test.csv")
test_df.head()

,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,...,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
0,5,88.936744,51.227894,70.358975,34.902572,1.445824,1.467243,122.90607,14.729974,46.482335,...,2.099846,2.352158,2.082630,1.589027,1.289155,1,1.009217,0.489898,0.299795,61.0
1,5,89.337180,53.389543,70.560647,35.669394,1.444537,1.425029,124.90825,17.776145,46.917667,...,2.115830,2.491462,2.086660,1.564957,1.307861,2,0.957529,0.800000,0.423832,28.2
2,4,76.517718,57.468202,59.310096,35.985997,1.197273,0.909037,122.90607,37.745630,44.289459,...,2.277143,2.213364,2.237858,1.368922,1.011105,1,1.125714,0.433013,0.447588,14.0
3,5,83.244760,58.211883,59.506081,35.973651,1.311510,1.407001,192.98100,22.872726,67.233835,...,2.383509,2.402249,2.248543,1.519383,1.396410,3,0.849473,1.200000,1.001723,81.0
4,5,61.357238,51.028500,26.270151,33.490036,1.217667,1.291771,136.31906,21.119847,49.466135,...,2.061562,1.888175,2.041492,1.557113,1.245082,2,1.030396,0.632456,0.297579,72.5


In [5]:
with open(processed / "metadata.json") as f:
    metadata = json.load(f)

print(metadata)

{'num_cols': ['number_of_elements', 'mean_atomic_mass', 'wtd_mean_atomic_mass', 'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass', 'wtd_entropy_atomic_mass', 'range_atomic_mass', 'wtd_range_atomic_mass', 'std_atomic_mass', 'wtd_std_atomic_mass', 'mean_fie', 'wtd_mean_fie', 'gmean_fie', 'wtd_gmean_fie', 'entropy_fie', 'wtd_entropy_fie', 'range_fie', 'wtd_range_fie', 'std_fie', 'wtd_std_fie', 'mean_atomic_radius', 'wtd_mean_atomic_radius', 'gmean_atomic_radius', 'wtd_gmean_atomic_radius', 'entropy_atomic_radius', 'wtd_entropy_atomic_radius', 'range_atomic_radius', 'wtd_range_atomic_radius', 'std_atomic_radius', 'wtd_std_atomic_radius', 'mean_Density', 'wtd_mean_Density', 'gmean_Density', 'wtd_gmean_Density', 'entropy_Density', 'wtd_entropy_Density', 'range_Density', 'wtd_range_Density', 'std_Density', 'wtd_std_Density', 'mean_ElectronAffinity', 'wtd_mean_ElectronAffinity', 'gmean_ElectronAffinity', 'wtd_gmean_ElectronAffinity', 'entropy_ElectronAffinity', 'wtd_entropy_E

In [6]:
X = train_df.drop(columns=metadata["target_col"])
y = train_df[metadata["target_col"]].to_numpy().astype(np.float32)

processor = make_processor(metadata)
n_num_cols = len(metadata["num_cols"])

In [7]:
best_trial = fit_xrfm_cross_validation(
    X, y, processor, n_num_cols, task=cfg["task"], device=device
)

print(f"Best params: {best_trial.params}")
print(f"Best MSE: {best_trial.value}")

[I 2026-04-22 23:30:06,324] A new study created in memory with name: no-name-a2eb6b48-15ef-42b0-af68-9d8fc7980e05


None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 13608, d: 81, and nval: 3402


Using cheap batch size
Optimal M batch size: 2661


...

Using expensive batch size
Optimal M batch size: 1289


Time taken for round 4: 2.382873296737671 seconds


Building trees:   0%|          | 0/1 [00:12<?, ?it/s]


[I 2026-04-23 00:36:13,651] Trial 99 finished with value: 90.44933013916015 and parameters: {'exponent': 0.9911187709784457, 'kernel_type': 'kpq', 'bandwidth': 4.017454407390906, 'diag': True, 'norm_p': 1.7273111514537483, 'reg': 0.015172336519636856}. Best is trial 74 with value: 88.57009735107422.


Tree has no split, stopping training
Using hard routing for tree prediction
Best params: {'exponent': 1.0356639820698017, 'kernel_type': 'kpq', 'bandwidth': 12.508264327921696, 'diag': True, 'norm_p': 1.5212010907067888, 'reg': 0.01550573055035075}
Best MSE: 88.57009735107422


In [8]:
model, pipeline, fit_time, train_metrics, val_metrics = fit_best_xrfm(
    X,
    y,
    best_trial,
    processor,
    n_num_cols,
    task=cfg["task"],
    device=device,
)

print(f"Training time: {fit_time:.2f}s")
print(f"Train RMSE: {train_metrics['rmse']:.4f}")
print(f"Val RMSE:   {val_metrics['rmse']:.4f}")

joblib.dump(model, MODELS_DIR / data_name / "xrfm.joblib")
joblib.dump(pipeline, MODELS_DIR / data_name / "xrfm_pipeline.joblib")

None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 13608, d: 81, and nval: 3402


Using expensive batch size
Optimal M batch size: 1289


Time taken for round 0: 2.370130777359009 seconds


...

Time taken for round 4: 2.377072334289551 seconds


Building trees:   0%|          | 0/1 [00:15<?, ?it/s]

Tree has no split, stopping training
Using hard routing for tree prediction


Using hard routing for tree prediction
Training time: 12.90s
Train RMSE: 5.0208
Val RMSE:   9.4292


['/home/shado/college_project/comp9417/9417GroupProject/models/superconductor/xrfm_pipeline.joblib']

In [9]:
model = joblib.load(MODELS_DIR / data_name / "xrfm.joblib")
pipeline = joblib.load(MODELS_DIR / data_name / "xrfm_pipeline.joblib")

In [10]:
X_test = test_df.drop(columns=metadata["target_col"])
y_test = test_df[metadata["target_col"]].to_numpy().astype(np.float32)

X_test = pipeline.transform(X_test)
t0 = time.perf_counter()
y_pred = model.predict(X_test)
infer_time = (time.perf_counter() - t0) / len(X_test)

Using hard routing for tree prediction


In [11]:
results = evaluate_regression(y_test, y_pred)
print(f"RMSE: {results['rmse']:.4f}")
print(f"Inference time per sample: {infer_time * 1e6:.2f}µs")

RMSE: 9.8139
Inference time per sample: 50.30µs


In [12]:
output = {
    **results,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
    "fit_time_s": fit_time,
    "infer_time_per_sample_s": infer_time,
    "best_params": {
        "tuning_metric": model.tuning_metric,
        "split_method": model.split_method,
        "max_leaf_size": model.max_leaf_size,
        **model.rfm_params,
    },
}

results_dir = RESULTS_BENCHMARK / "superconductor"
with open(results_dir / "xrfm.json", "w") as f:
    json.dump(output, f, indent=4)

print(f"Results saved to {results_dir / 'xrfm.json'}")

Results saved to /home/shado/college_project/comp9417/9417GroupProject/results/superconductor/xrfm.json
